In [ ]:
# !pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
# !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
# !pip install seaborn

## Configuración:
* Importación Librerias
* Definición rutas
* Definición funciones auxiliares

In [ ]:
# Reload modulos automaticamente
%load_ext autoreload
%autoreload 2


from modules.gnn_models import (
    GCN2L,
    GraphSAGE2L,
    GAT2L,
    GCN3L,
    GraphSAGE3L,
    GAT3L,
    GCNSampler2L,
    GraphSAGESample2L,
    GATSample2L
)

from modules.gnn import GNN
import itertools
import csv
import os
import torch_geometric
from torch_geometric.utils import degree
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
)

print("PyG:", torch_geometric.__version__, " | PyTorch:", torch.__version__)
TOR_LABELS_DICT = {'P2P': 0, 'C2P': 1, 'P2C': 2}

### Funciones Auxiliares plot

In [ ]:
def plot_confusion_matrix(cm,
                          classes,
                          normalize=False,
                          title='Confusion matrix',
                          fname=None,
                          cmap=plt.cm.Blues,
                          save=False):
    """
    Print and plot the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix (not normalized)')

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], fmt),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    
    if save and fname is not None:
        plt.savefig(f"{fname}.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix_2(cm,
                            classes,
                            normalize=False,
                            title='Confusion matrix',
                            fname=None,
                            cmap=plt.cm.Blues,
                            save=False):
    """
    Plot confusion matrix.
    If `normalize=True`, values are shown in percentage (%).
    """
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100  # -> %
        print("Normalized confusion matrix (%)")
        fmt  = ".1f"
        suffix = "%"
    else:
        print("Confusion matrix (not normalized)")
        fmt  = "d"
        suffix = ""

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45)
    plt.yticks(ticks, classes)

    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i,
                     f"{format(cm[i, j], fmt)}{suffix}",
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()

    # if save and fname:
        # plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
    plt.show()

## Entrenamiento

In [ ]:
from pathlib import Path

# Archivos del nuevo formato de grafo (nodos/aristas)
data_path = Path("/media/vale/KINGSTON/as-level-internet-graph/grafos")
nodes_file = data_path / "nodes_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv"
edges_file = data_path / "edges_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv"

# nodes_file = data_path / "nodes_rib_20260301_0000_to_20260302_0000_pruned_enriched_tesis.csv"
# edges_file = data_path / "edges_rib_20260301_0000_to_20260302_0000_pruned_enriched_tesis.csv"

if not nodes_file.exists():
    raise FileNotFoundError(f"No existe el archivo de nodos: {nodes_file}")
if not edges_file.exists():
    raise FileNotFoundError(f"No existe el archivo de aristas: {edges_file}")

print(f"Nodos: {nodes_file}")
print(f"Aristas: {edges_file}")



In [ ]:
import time



GLOBAL_CONFIG = {
'lr': 0.001,
'hidden_dim': 256,
'out_dim': 128,
'epochs': 500,
'patience': 120,
'weight_decay': 5e-4,
'dropout': 0.3,
'label_smooth': 0.05,
'grad_clip': 2.0,
'num_heads': 2,
'batch_size': 1024, #4
'num_neighbors': 15, # Para GraphSAGE/GAT con muestreo
'focal_gamma': 2.0, # Focal loss gamma
}


In [ ]:
import torch
import torch.nn.functional as F

def focal_loss(logits, targets, alpha=None, gamma=2.0, reduction='mean'):
    """Focal loss multiclase. gamma=0 equivale a cross-entropy pesada."""
    ce = F.cross_entropy(logits, targets, weight=alpha, reduction='none')
    with torch.no_grad():
        probs = F.softmax(logits, dim=1)
        p_t   = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        w     = (1.0 - p_t) ** gamma
    loss = w * ce
    if reduction == 'mean':  return loss.mean()
    if reduction == 'sum':   return loss.sum()
    return loss


In [ ]:
# from pathlib import Path

# # Archivos del nuevo formato de grafo (nodos/aristas)
# nodes_file = "/media/vale/KINGSTON/TESIS/data/RIBs/nodes_sanitized_rib_marzo_2024_enriched_tesis.csv"
# edges_file = "/media/vale/KINGSTON/TESIS/data/RIBs/edges_sanitized_rib_marzo_2024_enriched_tesis.csv"

# if not Path(nodes_file).exists():
#     raise FileNotFoundError(f"No existe el archivo de nodos: {nodes_file}")
# if not Path(edges_file).exists():
#     raise FileNotFoundError(f"No existe el archivo de aristas: {edges_file}")

# print(f"Nodos: {nodes_file}")
# print(f"Aristas: {edges_file}")


## Caso 0: Se utiliza grafo con atributos de nodos extraídos de PeeringDB

In [ ]:
# 1. Load data with the new format (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))

gnn.split_edges_classification(
    train_size=0.7,
    val_size=0.15,
    seed=42,
    balance_mode="proportional",  # usa todos los datos; class_weights compensan el desbalance
)

g0 = gnn.graph
print("\n[Loaded graph summary]")
print(f"- Nodes: {g0.num_nodes:,}")
print(f"- Edges: {g0.num_edges:,}")
print(f"- Feature dimension: {g0.x.shape[1]}")
print(f"- Edge data keys: {[k for k in g0.keys() if g0.is_edge_attr(k)]}")
print(f"- Train edges: {g0.train_mask.sum().item():,}")
print(f"- Validation edges: {g0.val_mask.sum().item():,}")
print(f"- Test edges: {g0.test_mask.sum().item():,}")
if getattr(g0, "edge_label", None) is not None:
    print(f"- Global label distribution: {Counter(g0.edge_label.tolist())}")
    tr = g0.edge_label[g0.train_mask]
    va = g0.edge_label[g0.val_mask]
    te = g0.edge_label[g0.test_mask]
    print(f"- Train distribution: {Counter(tr.tolist())}")
    print(f"- Validation distribution: {Counter(va.tolist())}")
    print(f"- Test distribution: {Counter(te.tolist())}")

    # 10 seg

In [ ]:
models = {
    # 'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}

# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }


### Encoder : decodeBilinear

In [ ]:

# # 1.- Definición parámetros (usar config global)
# # ----------------------------------------------
# in_feats      = gnn.graph.x.shape[1]
# hidden_feats  = GLOBAL_CONFIG['hidden_dim']
# emb_dim       = GLOBAL_CONFIG['out_dim']
# lr            = GLOBAL_CONFIG['lr']
# device        = "cuda" if torch.cuda.is_available() else "cpu"

# # 2.- Datos y máscaras
# # ----------------------------------------------
# g       = gnn.graph.to(device)
# feat    = g.x.to(device)
# edge_index = g.edge_index.to(device)
# edge_attr  = g.edge_attr.to(device) if getattr(g, "edge_attr", None) is not None else None
# g.log_out_deg = torch.log1p(degree(g.edge_index[0], g.num_nodes).float()).to(device)
# g.log_in_deg  = torch.log1p(degree(g.edge_index[1], g.num_nodes).float()).to(device)




In [ ]:

# label_key = next((k for k in ["edge_label", "label", "relationship", "Relationship"] if getattr(g, k, None) is not None), None)
# if label_key is None:
#     raise KeyError(f"Label column not found in graph. Available keys: {[k for k in g.keys() if g.is_edge_attr(k)]}")
# labels = getattr(g, label_key).long().to(device)

# train_m = g.train_mask.bool().to(device)
# val_m   = g.val_mask.bool().to(device) if getattr(g, "val_mask", None) is not None else None
# test_m  = g.test_mask.bool().to(device)

# if train_m.sum().item() == 0 or test_m.sum().item() == 0:
#     raise ValueError("train_mask/test_mask are empty. Check split_edges_classification().")
# if val_m is None or val_m.sum().item() == 0:
#     raise ValueError("val_mask is empty/not found. Use split_edges_classification with val_size > 0.")

# labeled_m = train_m | val_m | test_m
# n_classes = int(labels[labeled_m].max().item()) + 1

# # Class weights to reduce collapse into a dominant class
# class_counts = torch.bincount(labels[train_m], minlength=n_classes).float()
# class_weights = (class_counts.sum() / (n_classes * class_counts.clamp(min=1.0))).to(device)

# default_names = ["P2P", "C2P", "P2C"]
# class_names = default_names[:n_classes] if n_classes <= len(default_names) else [f"Class {i}" for i in range(n_classes)]

# print(f"Using device: {device}")
# print(f"Using edge label key: {label_key}")
# print(f"Number of classes detected: {n_classes}")
# print(f"Train class counts: {class_counts.tolist()}")
# print(f"Class weights: {class_weights.tolist()}")

# # 3.- Load node_id -> asn mapping for readable inference export
# id_to_asn = {}
# with open(nodes_file, newline="") as f:
#     reader = csv.DictReader(f)
#     for row in reader:
#         id_to_asn[int(row["node_id"])] = int(row["asn"])

# # 4.- Output directories
# output_root = Path("./results/end_to_end")
# plot_dir = output_root / "plots"
# pred_dir = output_root / "predictions"
# plot_dir.mkdir(parents=True, exist_ok=True)
# pred_dir.mkdir(parents=True, exist_ok=True)
# run_tag = edges_file.stem.replace("edges_", "")

# # 5.- Training per model (Bilinear Decoder)
# for name, Model in models.items():
#     print(f"\nTraining {name} (Bilinear)\n" + "-" * 40)

#     history = {"loss_tr": [], "acc_tr": [], "loss_val": [], "acc_val": [], "f1_val": []}
#     extra_kw = {'num_heads': GLOBAL_CONFIG['num_heads']} if 'GAT' in name else {}
#     model = Model(in_feats,
#                   GLOBAL_CONFIG['hidden_dim'],
#                   GLOBAL_CONFIG['out_dim'],
#                   out_feats_mlp=n_classes,
#                   drop=GLOBAL_CONFIG['dropout'],
#                   **extra_kw).to(device)
    
#     opt = torch.optim.Adam(model.parameters(), lr=GLOBAL_CONFIG['lr'], weight_decay=GLOBAL_CONFIG['weight_decay'])
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)

#     best_macro_f1, best_val_loss, wait, best_w = -1.0, float("inf"), 0, None

#     t0 = time.perf_counter()
#     for epoch in range(1, GLOBAL_CONFIG['epochs'] + 1):
#         model.train()
#         h = model.encode(feat, edge_index)
#         logits = model.decodeBilinear(edge_index, h)


#         # loss = F.cross_entropy(logits[train_m], 
#         #                        labels[train_m], 
#         #                        weight=class_weights, 
#         #                        label_smoothing=GLOBAL_CONFIG['label_smooth'])
        

#         loss = F.cross_entropy(logits[train_m], 
#                                labels[train_m], 

#                                label_smoothing=GLOBAL_CONFIG['label_smooth'])
        

#         opt.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CONFIG['grad_clip'])
#         opt.step()
#         scheduler.step(epoch - 1 + (1 / GLOBAL_CONFIG['epochs']))

#         with torch.no_grad():
#             preds_tr = logits[train_m].argmax(1)
#             acc_tr = (preds_tr == labels[train_m]).float().mean().item()

#         model.eval()
#         with torch.no_grad():
#             logits_val = model.decodeBilinear(edge_index, model.encode(feat, edge_index))

#             # loss_val = F.cross_entropy(logits_val[val_m], 
#             #                            labels[val_m], 
#             #                            weight=class_weights
#             #                            )
            
#             loss_val = F.cross_entropy(logits_val[val_m], 
#                                        labels[val_m])


#             preds_val = logits_val[val_m].argmax(1)
#             val_true = labels[val_m]
#             acc_val = (preds_val == val_true).float().mean().item()
#             macro_f1_val = f1_score(
#                 val_true.cpu().numpy(),
#                 preds_val.cpu().numpy(),
#                 average="macro",
#                 zero_division=0,
#             )

#         history["loss_tr"].append(loss.item())
#         history["acc_tr"].append(acc_tr * 100)
#         history["loss_val"].append(loss_val.item())
#         history["acc_val"].append(acc_val * 100)
#         history["f1_val"].append(macro_f1_val * 100)

#         improved_f1 = macro_f1_val > (best_macro_f1 + 1e-4)
#         tie_break_loss = abs(macro_f1_val - best_macro_f1) <= 1e-4 and loss_val.item() < (best_val_loss - 1e-4)

#         if improved_f1 or tie_break_loss:
#             best_macro_f1, best_val_loss, wait = macro_f1_val, loss_val.item(), 0
#             best_w = model.state_dict()
#         else:
#             wait += 1
#             if wait >= GLOBAL_CONFIG['patience']:
#                 print(f"Early stop at epoch {epoch} (best val macro-F1={best_macro_f1*100:.2f}%)")
#                 break

#         if epoch % 10 == 0 or epoch == 1:
#             print(
#                 f"ep{epoch:03d} | loss_tr {loss:.4f} acc_tr {acc_tr:.2%} | "
#                 f"loss_val {loss_val:.4f} acc_val {acc_val:.2%} f1_val {macro_f1_val:.2%}"
#             )

#     tiempo_total = time.perf_counter() - t0
#     mins, secs = divmod(tiempo_total, 60)
#     print(f"\n{'='*50}")
#     print(f"[TIEMPO] {name} (Bilinear): {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
#     print(f"{'='*50}")

#     if best_w is not None:
#         model.load_state_dict(best_w)

#     with torch.no_grad():
#         logits_best = model.decodeBilinear(edge_index, model.encode(feat, edge_index))
#         preds_best = logits_best[test_m].argmax(1).cpu()
#         truth = labels[test_m].cpu()

#     cm = confusion_matrix(truth, preds_best, labels=list(range(n_classes)))
#     plot_confusion_matrix_2(cm, classes=class_names, normalize=False, title=f"Confusion Matrix - {name}")
#     plot_confusion_matrix_2(cm, classes=class_names, normalize=True, title=f"Confusion Matrix (%) - {name}")

#     print(classification_report(truth, preds_best, digits=4, zero_division=0, target_names=class_names))

#     # Baseline: always predict the majority train class
#     majority_class = int(torch.mode(labels[train_m]).values.item())
#     baseline_preds = torch.full_like(truth, fill_value=majority_class)
#     baseline_acc = accuracy_score(truth, baseline_preds) * 100
#     baseline_f1_macro = f1_score(truth, baseline_preds, average="macro", zero_division=0) * 100

#     prec_micro = precision_score(truth, preds_best, average="micro", zero_division=0) * 100
#     rec_micro  = recall_score(truth, preds_best, average="micro", zero_division=0) * 100
#     f1_micro   = f1_score(truth, preds_best, average="micro", zero_division=0) * 100
#     acc        = accuracy_score(truth, preds_best) * 100

#     prec_macro = precision_score(truth, preds_best, average="macro", zero_division=0) * 100
#     rec_macro  = recall_score(truth, preds_best, average="macro", zero_division=0) * 100
#     f1_macro   = f1_score(truth, preds_best, average="macro", zero_division=0) * 100

#     prec_w = precision_score(truth, preds_best, average="weighted", zero_division=0) * 100
#     rec_w  = recall_score(truth, preds_best, average="weighted", zero_division=0) * 100
#     f1_w   = f1_score(truth, preds_best, average="weighted", zero_division=0) * 100

#     recalls_per_class = recall_score(
#         truth,
#         preds_best,
#         average=None,
#         labels=list(range(n_classes)),
#         zero_division=0,
#     )

#     print(f"[BASELINE] Majority class={majority_class} ({class_names[majority_class] if majority_class < len(class_names) else majority_class})")
#     print(f"[BASELINE] Acc={baseline_acc:.2f}% | Macro-F1={baseline_f1_macro:.2f}%")
#     print(f"[MODEL]    Acc={acc:.2f}% | Macro-F1={f1_macro:.2f}%")
#     print(f"[TEST] micro:    Prec={prec_micro:.2f}%  Rec={rec_micro:.2f}%  F1={f1_micro:.2f}% | Acc={acc:.2f}%")
#     print(f"[TEST] macro:    Prec={prec_macro:.2f}%  Rec={rec_macro:.2f}%  F1={f1_macro:.2f}%")
#     print(f"[TEST] weighted: Prec={prec_w:.2f}%  Rec={rec_w:.2f}%  F1={f1_w:.2f}%")

#     print("Recall per class (test):")
#     for i, r in enumerate(recalls_per_class):
#         cname = class_names[i] if i < len(class_names) else f"Class {i}"
#         print(f"  - {cname}: {r*100:.2f}%")

#     epoch_axis = range(1, len(history["loss_tr"]) + 1)

#     plt.figure(figsize=(8, 4))
#     plt.plot(epoch_axis, history["loss_tr"], label="Train Loss")
#     plt.plot(epoch_axis, history["loss_val"], label="Validation Loss")
#     plt.xlabel("Epoch")
#     plt.ylabel("Loss")
#     plt.title(f"Loss Curve - {name}")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(plot_dir / f"loss_{name}_{run_tag}_bilinear.png", dpi=300, bbox_inches='tight')
#     plt.show()

#     plt.figure(figsize=(8, 4))
#     plt.plot(epoch_axis, history["acc_tr"], label="Train Accuracy (%)")
#     plt.plot(epoch_axis, history["acc_val"], label="Validation Accuracy (%)")
#     plt.xlabel("Epoch")
#     plt.ylabel("Accuracy (%)")
#     plt.title(f"Accuracy Curve - {name}")
#     plt.ylim(0, 100)
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(plot_dir / f"accuracy_{name}_{run_tag}_bilinear.png", dpi=300, bbox_inches='tight')
#     plt.show()

#     plt.figure(figsize=(8, 4))
#     plt.plot(epoch_axis, history["f1_val"], label="Validation Macro-F1 (%)", color="purple")
#     plt.xlabel("Epoch")
#     plt.ylabel("Macro-F1 (%)")
#     plt.title(f"Validation Macro-F1 Curve - {name}")
#     plt.ylim(0, 100)
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(plot_dir / f"macrof1_val_{name}_{run_tag}_bilinear.png", dpi=300, bbox_inches='tight')
#     plt.show()

#     out_file = pred_dir / f"inference_results_{name}_{run_tag}_bilinear.txt"
#     u, v = g.edge_index[0], g.edge_index[1]
#     test_eids = torch.where(test_m)[0].cpu().tolist()
#     with open(out_file, "w") as f:
#         f.write("src_id|dst_id|src_asn|dst_asn|pred|real\n")
#         for i, eidx in enumerate(test_eids):
#             src_id = int(u[eidx].item())
#             dst_id = int(v[eidx].item())
#             src_asn = id_to_asn.get(src_id, -1)
#             dst_asn = id_to_asn.get(dst_id, -1)
#             pred = int(preds_best[i].item())
#             real = int(truth[i].item())
#             f.write(f"{src_id}|{dst_id}|{src_asn}|{dst_asn}|{pred}|{real}\n")

#     print(f"Inference file saved at: {out_file}")

### Encoder : MLP

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
import matplotlib.pyplot as plt

# Global parameters (use unified config)
in_feats  = gnn.graph.x.shape[1]
device    = "cuda" if torch.cuda.is_available() else "cpu"
lr        = GLOBAL_CONFIG['lr']
patience  = GLOBAL_CONFIG['patience']
num_epochs = GLOBAL_CONFIG['epochs']

# Data and masks
g       = gnn.graph.to(device)
feat    = g.x.to(device)
edge_index = g.edge_index.to(device)
edge_attr  = g.edge_attr.to(device) if getattr(g, "edge_attr", None) is not None else None

# Degree features para forzar la asimetría C2P vs P2C en el decoder
# g.log_out_deg = torch.log1p(degree(g.edge_index[0], g.num_nodes).float()).to(device)
# g.log_in_deg  = torch.log1p(degree(g.edge_index[1], g.num_nodes).float()).to(device)
label_key = next((k for k in ["edge_label", "label", "relationship", "Relationship"] if getattr(g, k, None) is not None), None)
if label_key is None:
    raise KeyError(f"Label column not found in graph. Available keys: {[k for k in g.keys() if g.is_edge_attr(k)]}")

labels  = getattr(g, label_key).long().to(device)
train_m = g.train_mask.bool().to(device)
val_m   = g.val_mask.bool().to(device) if getattr(g, "val_mask", None) is not None else None
test_m  = g.test_mask.bool().to(device)

if val_m is None or val_m.sum().item() == 0:
    raise ValueError("val_mask is empty/not found. Use split_edges_classification with val_size > 0.")

labeled_m = train_m | val_m | test_m
n_classes = int(labels[labeled_m].max().item()) + 1

class_counts = torch.bincount(labels[train_m], minlength=n_classes).float()
class_weights = (class_counts.sum() / (n_classes * class_counts.clamp(min=1.0))).to(device)

default_names = ["P2P", "C2P", "P2C"]
class_names = default_names[:n_classes] if n_classes <= len(default_names) else [f"Class {i}" for i in range(n_classes)]

for name, Model in models.items():
    print(f"\n=== Training {name} (MLP Decoder) ===")


    extra_kw = {'num_heads': GLOBAL_CONFIG['num_heads']} if 'GAT' in name else {}
    model = Model(in_feats,
                  GLOBAL_CONFIG['hidden_dim'],
                  GLOBAL_CONFIG['out_dim'],
                  out_feats_mlp=n_classes,
                  edge_dim=1,
                  drop=GLOBAL_CONFIG['dropout'],
                  **extra_kw).to(device)
    
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=GLOBAL_CONFIG['weight_decay'])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=100, T_mult=2)

    hist = {"loss_tr": [], "acc_tr": [], "loss_val": [], "acc_val": [], "f1_val": []}
    best_macro_f1, best_loss, wait, best_w = -1.0, float("inf"), 0, None

    t0 = time.perf_counter()
    for ep in range(1, GLOBAL_CONFIG['epochs'] + 1):
        model.train()
        h = model.encode(feat, edge_index)
        logits = model.decodeMLP(edge_index, h, edge_attr)

        loss = focal_loss(logits[train_m], labels[train_m],
                          alpha=class_weights,
                          gamma=GLOBAL_CONFIG['focal_gamma'])
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CONFIG['grad_clip'])
        opt.step()
        scheduler.step(ep - 1 + (1 / num_epochs))

        with torch.no_grad():
            preds_tr = logits[train_m].argmax(1)
            acc_tr = (preds_tr == labels[train_m]).float().mean().item()

        model.eval()
        with torch.no_grad():
            logits_t = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)

            loss_val = F.cross_entropy(logits_t[val_m], 
                                       labels[val_m], 
                                       weight=class_weights)
            
            # loss_val = F.cross_entropy(logits_t[val_m], 
            #                            labels[val_m])

            preds_val = logits_t[val_m].argmax(1)
            val_true = labels[val_m]
            acc_val = (preds_val == val_true).float().mean().item()
            macro_f1_val = f1_score(
                val_true.cpu().numpy(),
                preds_val.cpu().numpy(),
                average="macro",
                zero_division=0,
            )

        hist["loss_tr"].append(loss.item())
        hist["acc_tr"].append(acc_tr * 100)
        hist["loss_val"].append(loss_val.item())
        hist["acc_val"].append(acc_val * 100)
        hist["f1_val"].append(macro_f1_val * 100)

        improved_f1 = macro_f1_val > (best_macro_f1 + 1e-4)
        tie_break_loss = abs(macro_f1_val - best_macro_f1) <= 1e-4 and loss_val.item() < (best_loss - 1e-4)

        if improved_f1 or tie_break_loss:
            best_macro_f1, best_loss, wait = macro_f1_val, loss_val.item(), 0
            best_w = model.state_dict()
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stop at epoch {ep} (best val macro-F1={best_macro_f1*100:.2f}%)")
                break

        if ep % 10 == 0 or ep == 1:
            print(
                f"ep{ep:03d} | loss_tr {loss:.4f} acc_tr {acc_tr:.2%} | "
                f"loss_val {loss_val:.4f} acc_val {acc_val:.2%} f1_val {macro_f1_val:.2%}"
            )

    tiempo_total = time.perf_counter() - t0
    mins, secs = divmod(tiempo_total, 60)
    print(f"\n{'='*50}")
    print(f"[TIEMPO] {name} (MLP): {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
    print(f"{'='*50}")

    if best_w is not None:
        model.load_state_dict(best_w)

    with torch.no_grad():
        logits_best = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
        preds_best = logits_best[test_m].argmax(1).cpu()
        truth = labels[test_m].cpu()

    cm = confusion_matrix(truth, preds_best, labels=list(range(n_classes)))
    plot_confusion_matrix_2(cm, class_names, False, title=f"Confusion - {name}")
    plot_confusion_matrix_2(cm, class_names, True, title=f"Confusion (%) - {name}")

    print(classification_report(truth, preds_best, digits=4, zero_division=0, target_names=class_names))

    majority_class = int(torch.mode(labels[train_m]).values.item())
    baseline_preds = torch.full_like(truth, fill_value=majority_class)
    baseline_acc = accuracy_score(truth, baseline_preds) * 100
    baseline_f1_macro = f1_score(truth, baseline_preds, average="macro", zero_division=0) * 100

    recalls_per_class = recall_score(
        truth,
        preds_best,
        average=None,
        labels=list(range(n_classes)),
        zero_division=0,
    )

    model_acc = accuracy_score(truth, preds_best) * 100
    model_f1_macro = f1_score(truth, preds_best, average="macro", zero_division=0) * 100

    print(f"[BASELINE] Majority class={majority_class} ({class_names[majority_class] if majority_class < len(class_names) else majority_class})")
    print(f"[BASELINE] Acc={baseline_acc:.2f}% | Macro-F1={baseline_f1_macro:.2f}%")
    print(f"[MODEL]    Acc={model_acc:.2f}% | Macro-F1={model_f1_macro:.2f}%")

    print("Recall per class (test):")
    for i, r in enumerate(recalls_per_class):
        cname = class_names[i] if i < len(class_names) else f"Class {i}"
        print(f"  - {cname}: {r*100:.2f}%")

    epochs_done = range(1, len(hist["loss_tr"]) + 1)

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_done, hist["loss_tr"], label="Train Loss")
    plt.plot(epochs_done, hist["loss_val"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss - {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_done, hist["acc_tr"], label="Train Accuracy (%)")
    plt.plot(epochs_done, hist["acc_val"], label="Validation Accuracy (%)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    plt.title(f"Accuracy - {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_done, hist["f1_val"], label="Validation Macro-F1 (%)", color="purple")
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1 (%)")
    plt.ylim(0, 100)
    plt.title(f"Validation Macro-F1 - {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ── Overfitting check ─────────────────────────────────────────────
    print("\n[OVERFITTING CHECK]")

    # 1. Gap train vs val (al final del entrenamiento)
    final_acc_tr  = hist["acc_tr"][-1]
    final_acc_val = hist["acc_val"][-1]
    gap = final_acc_tr - final_acc_val
    print(f"  Train acc final : {final_acc_tr:.2f}%")
    print(f"  Val   acc final : {final_acc_val:.2f}%")
    print(f"  Gap             : {gap:.2f}%  {'⚠ posible overfitting' if gap > 10 else '✓ OK'}")

    # 2. Val F1 vs Test F1 (deben ser cercanos si el split es representativo)
    f1_gap = abs(best_macro_f1 * 100 - model_f1_macro)
    print(f"  Best val macro-F1 : {best_macro_f1*100:.2f}%")
    print(f"  Test macro-F1     : {model_f1_macro:.2f}%")
    print(f"  Diferencia        : {f1_gap:.2f}%  {'⚠ split sospechoso' if f1_gap > 5 else '✓ OK'}")

    # 3. Sanity check: predicción con etiquetas mezcladas en train
    # Si el modelo aprendió patrones reales, al mezclar las etiquetas del
    # train set y re-evaluar sobre esas mismas aristas, el score debe caer
    # cerca del azar (~33% para 3 clases).
    with torch.no_grad():
        logits_tr = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
        preds_tr_final = logits_tr[train_m].argmax(1).cpu()
        real_labels_tr = labels[train_m].cpu()
        shuffled_labels_tr = real_labels_tr[torch.randperm(len(real_labels_tr))]

    acc_real     = accuracy_score(real_labels_tr, preds_tr_final)
    acc_shuffled = accuracy_score(shuffled_labels_tr, preds_tr_final)
    expected_random = 1.0 / n_classes
    print(f"\n  Acc sobre train (etiquetas reales)   : {acc_real*100:.2f}%")
    print(f"  Acc sobre train (etiquetas mezcladas): {acc_shuffled*100:.2f}%")
    print(f"  Azar esperado                        : {expected_random*100:.1f}%")
    print(f"  {'✓ El modelo usa patrones reales' if acc_shuffled < expected_random + 0.05 else '⚠ Posible memorización'}")



## Caso 1: Se utiliza grafo con atributos de centralidad

In [ ]:
# 1. Load data with the new format (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset_only_cntrality_attr(str(nodes_file), str(edges_file))

gnn.split_edges_classification(
    train_size=0.8,
    val_size=0.15,
    seed=42,
    balance_mode="strict_equal",   # use "proportional" to keep natural class distribution
)

g0 = gnn.graph
print("\n[Loaded graph summary]")
print(f"- Nodes: {g0.num_nodes:,}")
print(f"- Edges: {g0.num_edges:,}")
print(f"- Feature dimension: {g0.x.shape[1]}")
print(f"- Edge data keys: {[k for k in g0.keys() if g0.is_edge_attr(k)]}")
print(f"- Train edges: {g0.train_mask.sum().item():,}")
print(f"- Validation edges: {g0.val_mask.sum().item():,}")
print(f"- Test edges: {g0.test_mask.sum().item():,}")

if getattr(g0, "edge_label", None) is not None:
    print(f"- Global label distribution: {Counter(g0.edge_label.tolist())}")
    tr = g0.edge_label[g0.train_mask]
    va = g0.edge_label[g0.val_mask]
    te = g0.edge_label[g0.test_mask]
    print(f"- Train distribution: {Counter(tr.tolist())}")
    print(f"- Validation distribution: {Counter(va.tolist())}")
    print(f"- Test distribution: {Counter(te.tolist())}")


In [ ]:
# Modelos a entrenar
models = {
    # 'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}

# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }



models_dict = models  # alias por compatibilidad

### Decode: DotProduct

In [ ]:
# 1.- Definición parámetros (usar config global)
# ----------------------------------------------
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
emb_dim       = GLOBAL_CONFIG['out_dim']
lr            = GLOBAL_CONFIG['lr']
device        = "cuda" if torch.cuda.is_available() else "cpu"

# 2.- Datos y máscaras
# ----------------------------------------------
g       = gnn.graph.to(device)
feat    = g.x.to(device)
edge_index = g.edge_index.to(device)
edge_attr  = g.edge_attr.to(device) if getattr(g, "edge_attr", None) is not None else None

In [ ]:

label_key = next((k for k in ["edge_label", "label", "relationship", "Relationship"] if getattr(g, k, None) is not None), None)
if label_key is None:
    raise KeyError(f"Label column not found in graph. Available keys: {[k for k in g.keys() if g.is_edge_attr(k)]}")
labels = getattr(g, label_key).long().to(device)

train_m = g.train_mask.bool().to(device)
val_m   = g.val_mask.bool().to(device) if getattr(g, "val_mask", None) is not None else None
test_m  = g.test_mask.bool().to(device)

if train_m.sum().item() == 0 or test_m.sum().item() == 0:
    raise ValueError("train_mask/test_mask are empty. Check split_edges_classification().")
if val_m is None or val_m.sum().item() == 0:
    raise ValueError("val_mask is empty/not found. Use split_edges_classification with val_size > 0.")

labeled_m = train_m | val_m | test_m
n_classes = int(labels[labeled_m].max().item()) + 1

# Class weights to reduce collapse into a dominant class
class_counts = torch.bincount(labels[train_m], minlength=n_classes).float()
class_weights = (class_counts.sum() / (n_classes * class_counts.clamp(min=1.0))).to(device)

default_names = ["P2P", "C2P", "P2C"]
class_names = default_names[:n_classes] if n_classes <= len(default_names) else [f"Class {i}" for i in range(n_classes)]

print(f"Using device: {device}")
print(f"Using edge label key: {label_key}")
print(f"Number of classes detected: {n_classes}")
print(f"Train class counts: {class_counts.tolist()}")
print(f"Class weights: {class_weights.tolist()}")

# 3.- Load node_id -> asn mapping for readable inference export
id_to_asn = {}
with open(nodes_file, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        id_to_asn[int(row["node_id"])] = int(row["asn"])

# 4.- Output directories
output_root = Path("./results/end_to_end")
plot_dir = output_root / "plots"
pred_dir = output_root / "predictions"
plot_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(parents=True, exist_ok=True)
run_tag = edges_file.stem.replace("edges_", "")

# 5.- Training per model (Bilinear Decoder)
for name, Model in models.items():
    print(f"\nTraining {name} (Bilinear)\n" + "-" * 40)

    history = {"loss_tr": [], "acc_tr": [], "loss_val": [], "acc_val": [], "f1_val": []}
    extra_kw = {'num_heads': GLOBAL_CONFIG['num_heads']} if 'GAT' in name else {}
    model = Model(in_feats,
                  GLOBAL_CONFIG['hidden_dim'],
                  GLOBAL_CONFIG['out_dim'],
                  out_feats_mlp=n_classes,
                  drop=GLOBAL_CONFIG['dropout'],
                  **extra_kw).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=GLOBAL_CONFIG['lr'], weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)

    best_macro_f1, best_val_loss, wait, best_w = -1.0, float("inf"), 0, None

    t0 = time.perf_counter()
    for epoch in range(1, GLOBAL_CONFIG['epochs'] + 1):
        model.train()
        h = model.encode(feat, edge_index)
        logits = model.decodeBilinear(edge_index, h)

        loss = F.cross_entropy(logits[train_m], labels[train_m],
                               weight=class_weights, label_smoothing=GLOBAL_CONFIG['label_smooth'])
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CONFIG['grad_clip'])
        opt.step()
        scheduler.step(epoch - 1 + (1 / GLOBAL_CONFIG['epochs']))

        with torch.no_grad():
            preds_tr = logits[train_m].argmax(1)
            acc_tr = (preds_tr == labels[train_m]).float().mean().item()

        model.eval()
        with torch.no_grad():
            logits_val = model.decodeBilinear(edge_index, model.encode(feat, edge_index))
            loss_val = F.cross_entropy(logits_val[val_m], labels[val_m], weight=class_weights)
            preds_val = logits_val[val_m].argmax(1)
            val_true = labels[val_m]
            acc_val = (preds_val == val_true).float().mean().item()
            macro_f1_val = f1_score(
                val_true.cpu().numpy(),
                preds_val.cpu().numpy(),
                average="macro",
                zero_division=0,
            )

        history["loss_tr"].append(loss.item())
        history["acc_tr"].append(acc_tr * 100)
        history["loss_val"].append(loss_val.item())
        history["acc_val"].append(acc_val * 100)
        history["f1_val"].append(macro_f1_val * 100)

        improved_f1 = macro_f1_val > (best_macro_f1 + 1e-4)
        tie_break_loss = abs(macro_f1_val - best_macro_f1) <= 1e-4 and loss_val.item() < (best_val_loss - 1e-4)

        if improved_f1 or tie_break_loss:
            best_macro_f1, best_val_loss, wait = macro_f1_val, loss_val.item(), 0
            best_w = model.state_dict()
        else:
            wait += 1
            if wait >= GLOBAL_CONFIG['patience']:
                print(f"Early stop at epoch {epoch} (best val macro-F1={best_macro_f1*100:.2f}%)")
                break

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"ep{epoch:03d} | loss_tr {loss:.4f} acc_tr {acc_tr:.2%} | "
                f"loss_val {loss_val:.4f} acc_val {acc_val:.2%} f1_val {macro_f1_val:.2%}"
            )

    tiempo_total = time.perf_counter() - t0
    mins, secs = divmod(tiempo_total, 60)
    print(f"\n{'='*50}")
    print(f"[TIEMPO] {name} (Bilinear): {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
    print(f"{'='*50}")

    if best_w is not None:
        model.load_state_dict(best_w)

    with torch.no_grad():
        logits_best = model.decodeBilinear(edge_index, model.encode(feat, edge_index))
        preds_best = logits_best[test_m].argmax(1).cpu()
        truth = labels[test_m].cpu()

    cm = confusion_matrix(truth, preds_best, labels=list(range(n_classes)))
    plot_confusion_matrix_2(cm, classes=class_names, normalize=False, title=f"Confusion Matrix - {name}")
    plot_confusion_matrix_2(cm, classes=class_names, normalize=True, title=f"Confusion Matrix (%) - {name}")

    print(classification_report(truth, preds_best, digits=4, zero_division=0, target_names=class_names))

    # Baseline: always predict the majority train class
    majority_class = int(torch.mode(labels[train_m]).values.item())
    baseline_preds = torch.full_like(truth, fill_value=majority_class)
    baseline_acc = accuracy_score(truth, baseline_preds) * 100
    baseline_f1_macro = f1_score(truth, baseline_preds, average="macro", zero_division=0) * 100

    prec_micro = precision_score(truth, preds_best, average="micro", zero_division=0) * 100
    rec_micro  = recall_score(truth, preds_best, average="micro", zero_division=0) * 100
    f1_micro   = f1_score(truth, preds_best, average="micro", zero_division=0) * 100
    acc        = accuracy_score(truth, preds_best) * 100

    prec_macro = precision_score(truth, preds_best, average="macro", zero_division=0) * 100
    rec_macro  = recall_score(truth, preds_best, average="macro", zero_division=0) * 100
    f1_macro   = f1_score(truth, preds_best, average="macro", zero_division=0) * 100

    prec_w = precision_score(truth, preds_best, average="weighted", zero_division=0) * 100
    rec_w  = recall_score(truth, preds_best, average="weighted", zero_division=0) * 100
    f1_w   = f1_score(truth, preds_best, average="weighted", zero_division=0) * 100

    recalls_per_class = recall_score(
        truth,
        preds_best,
        average=None,
        labels=list(range(n_classes)),
        zero_division=0,
    )

    print(f"[BASELINE] Majority class={majority_class} ({class_names[majority_class] if majority_class < len(class_names) else majority_class})")
    print(f"[BASELINE] Acc={baseline_acc:.2f}% | Macro-F1={baseline_f1_macro:.2f}%")
    print(f"[MODEL]    Acc={acc:.2f}% | Macro-F1={f1_macro:.2f}%")
    print(f"[TEST] micro:    Prec={prec_micro:.2f}%  Rec={rec_micro:.2f}%  F1={f1_micro:.2f}% | Acc={acc:.2f}%")
    print(f"[TEST] macro:    Prec={prec_macro:.2f}%  Rec={rec_macro:.2f}%  F1={f1_macro:.2f}%")
    print(f"[TEST] weighted: Prec={prec_w:.2f}%  Rec={rec_w:.2f}%  F1={f1_w:.2f}%")

    print("Recall per class (test):")
    for i, r in enumerate(recalls_per_class):
        cname = class_names[i] if i < len(class_names) else f"Class {i}"
        print(f"  - {cname}: {r*100:.2f}%")

    epoch_axis = range(1, len(history["loss_tr"]) + 1)

    plt.figure(figsize=(8, 4))
    plt.plot(epoch_axis, history["loss_tr"], label="Train Loss")
    plt.plot(epoch_axis, history["loss_val"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss Curve - {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / f"loss_{name}_{run_tag}_bilinear.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(epoch_axis, history["acc_tr"], label="Train Accuracy (%)")
    plt.plot(epoch_axis, history["acc_val"], label="Validation Accuracy (%)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Accuracy Curve - {name}")
    plt.ylim(0, 100)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / f"accuracy_{name}_{run_tag}_bilinear.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(epoch_axis, history["f1_val"], label="Validation Macro-F1 (%)", color="purple")
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1 (%)")
    plt.title(f"Validation Macro-F1 Curve - {name}")
    plt.ylim(0, 100)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / f"macrof1_val_{name}_{run_tag}_bilinear.png", dpi=300, bbox_inches='tight')
    plt.show()

    out_file = pred_dir / f"inference_results_{name}_{run_tag}_bilinear.txt"
    u, v = g.edge_index[0], g.edge_index[1]
    test_eids = torch.where(test_m)[0].cpu().tolist()
    with open(out_file, "w") as f:
        f.write("src_id|dst_id|src_asn|dst_asn|pred|real\n")
        for i, eidx in enumerate(test_eids):
            src_id = int(u[eidx].item())
            dst_id = int(v[eidx].item())
            src_asn = id_to_asn.get(src_id, -1)
            dst_asn = id_to_asn.get(dst_id, -1)
            pred = int(preds_best[i].item())
            real = int(truth[i].item())
            f.write(f"{src_id}|{dst_id}|{src_asn}|{dst_asn}|{pred}|{real}\n")

    print(f"Inference file saved at: {out_file}")

# ── Overfitting check ─────────────────────────────────────────────
print("\n[OVERFITTING CHECK]")

# 1. Gap train vs val (al final del entrenamiento)
final_acc_tr  = hist["acc_tr"][-1]
final_acc_val = hist["acc_val"][-1]
gap = final_acc_tr - final_acc_val
print(f"  Train acc final : {final_acc_tr:.2f}%")
print(f"  Val   acc final : {final_acc_val:.2f}%")
print(f"  Gap             : {gap:.2f}%  {'⚠ posible overfitting' if gap > 10 else '✓ OK'}")

# 2. Val F1 vs Test F1 (deben ser cercanos si el split es representativo)
f1_gap = abs(best_macro_f1 * 100 - model_f1_macro)
print(f"  Best val macro-F1 : {best_macro_f1*100:.2f}%")
print(f"  Test macro-F1     : {model_f1_macro:.2f}%")
print(f"  Diferencia        : {f1_gap:.2f}%  {'⚠ split sospechoso' if f1_gap > 5 else '✓ OK'}")

# 3. Sanity check: predicción con etiquetas mezcladas en train
# Si el modelo aprendió patrones reales, al mezclar las etiquetas del
# train set y re-evaluar sobre esas mismas aristas, el score debe caer
# cerca del azar (~33% para 3 clases).
with torch.no_grad():
    logits_tr = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
    preds_tr_final = logits_tr[train_m].argmax(1).cpu()
    real_labels_tr = labels[train_m].cpu()
    shuffled_labels_tr = real_labels_tr[torch.randperm(len(real_labels_tr))]

acc_real     = accuracy_score(real_labels_tr, preds_tr_final)
acc_shuffled = accuracy_score(shuffled_labels_tr, preds_tr_final)
expected_random = 1.0 / n_classes
print(f"\n  Acc sobre train (etiquetas reales)   : {acc_real*100:.2f}%")
print(f"  Acc sobre train (etiquetas mezcladas): {acc_shuffled*100:.2f}%")
print(f"  Azar esperado                        : {expected_random*100:.1f}%")
print(f"  {'✓ El modelo usa patrones reales' if acc_shuffled < expected_random + 0.05 else '⚠ Posible memorización'}")


### Decode: MLP

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
import matplotlib.pyplot as plt

# Global parameters (use unified config)
in_feats  = gnn.graph.x.shape[1]
device    = "cuda" if torch.cuda.is_available() else "cpu"
lr        = GLOBAL_CONFIG['lr']
patience  = GLOBAL_CONFIG['patience']

# Data and masks
g       = gnn.graph.to(device)
feat    = g.x.to(device)
edge_index = g.edge_index.to(device)
edge_attr  = g.edge_attr.to(device) if getattr(g, "edge_attr", None) is not None else None

label_key = next((k for k in ["edge_label", "label", "relationship", "Relationship"] if getattr(g, k, None) is not None), None)
if label_key is None:
    raise KeyError(f"Label column not found in graph. Available keys: {[k for k in g.keys() if g.is_edge_attr(k)]}")

labels  = getattr(g, label_key).long().to(device)
train_m = g.train_mask.bool().to(device)
val_m   = g.val_mask.bool().to(device) if getattr(g, "val_mask", None) is not None else None
test_m  = g.test_mask.bool().to(device)

if val_m is None or val_m.sum().item() == 0:
    raise ValueError("val_mask is empty/not found. Use split_edges_classification with val_size > 0.")

labeled_m = train_m | val_m | test_m
n_classes = int(labels[labeled_m].max().item()) + 1

class_counts = torch.bincount(labels[train_m], minlength=n_classes).float()
class_weights = (class_counts.sum() / (n_classes * class_counts.clamp(min=1.0))).to(device)

default_names = ["P2P", "C2P", "P2C"]
class_names = default_names[:n_classes] if n_classes <= len(default_names) else [f"Class {i}" for i in range(n_classes)]

for name, Model in models.items():
    print(f"\n=== Training {name} (MLP Decoder) ===")

    # model = Model(in_feats, 
    #               GLOBAL_CONFIG['hidden_dim'], 
    #               GLOBAL_CONFIG['out_dim'], 
    #               out_feats_mlp=n_classes
    #               ).to(device)


    extra_kw = {'num_heads': GLOBAL_CONFIG['num_heads']} if 'GAT' in name else {}
    model = Model(in_feats,
                  GLOBAL_CONFIG['hidden_dim'],
                  GLOBAL_CONFIG['out_dim'],
                  out_feats_mlp=n_classes,
                  edge_dim=1,
                  drop=GLOBAL_CONFIG['dropout'],
                  **extra_kw).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=GLOBAL_CONFIG['lr'], weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=100, T_mult=2)

    hist = {"loss_tr": [], "acc_tr": [], "loss_val": [], "acc_val": [], "f1_val": []}
    best_macro_f1, best_loss, wait, best_w = -1.0, float("inf"), 0, None

    t0 = time.perf_counter()
    for ep in range(1, GLOBAL_CONFIG['epochs'] + 1):
        model.train()
        h = model.encode(feat, edge_index)
        logits = model.decodeMLP(edge_index, h, edge_attr)

        loss = focal_loss(logits[train_m], labels[train_m],
                          alpha=class_weights,
                          gamma=GLOBAL_CONFIG['focal_gamma'])
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CONFIG['grad_clip'])
        opt.step()
        scheduler.step(ep - 1 + (1 / GLOBAL_CONFIG['epochs']))

        with torch.no_grad():
            preds_tr = logits[train_m].argmax(1)
            acc_tr = (preds_tr == labels[train_m]).float().mean().item()

        model.eval()
        with torch.no_grad():
            logits_t = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
            loss_val = F.cross_entropy(logits_t[val_m], labels[val_m], weight=class_weights)
            preds_val = logits_t[val_m].argmax(1)
            val_true = labels[val_m]
            acc_val = (preds_val == val_true).float().mean().item()
            macro_f1_val = f1_score(
                val_true.cpu().numpy(),
                preds_val.cpu().numpy(),
                average="macro",
                zero_division=0,
            )

        hist["loss_tr"].append(loss.item())
        hist["acc_tr"].append(acc_tr * 100)
        hist["loss_val"].append(loss_val.item())
        hist["acc_val"].append(acc_val * 100)
        hist["f1_val"].append(macro_f1_val * 100)

        improved_f1 = macro_f1_val > (best_macro_f1 + 1e-4)
        tie_break_loss = abs(macro_f1_val - best_macro_f1) <= 1e-4 and loss_val.item() < (best_loss - 1e-4)

        if improved_f1 or tie_break_loss:
            best_macro_f1, best_loss, wait = macro_f1_val, loss_val.item(), 0
            best_w = model.state_dict()
        else:
            wait += 1
            if wait >= GLOBAL_CONFIG['patience']:
                print(f"Early stop at epoch {ep} (best val macro-F1={best_macro_f1*100:.2f}%)")
                break

        if ep % 10 == 0 or ep == 1:
            print(
                f"ep{ep:03d} | loss_tr {loss:.4f} acc_tr {acc_tr:.2%} | "
                f"loss_val {loss_val:.4f} acc_val {acc_val:.2%} f1_val {macro_f1_val:.2%}"
            )

    tiempo_total = time.perf_counter() - t0
    mins, secs = divmod(tiempo_total, 60)
    print(f"\n{'='*50}")
    print(f"[TIEMPO] {name} (MLP): {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
    print(f"{'='*50}")

    if best_w is not None:
        model.load_state_dict(best_w)

    with torch.no_grad():
        logits_best = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
        preds_best = logits_best[test_m].argmax(1).cpu()
        truth = labels[test_m].cpu()

    cm = confusion_matrix(truth, preds_best, labels=list(range(n_classes)))
    plot_confusion_matrix_2(cm, class_names, False, title=f"Confusion - {name}")
    plot_confusion_matrix_2(cm, class_names, True, title=f"Confusion (%) - {name}")

    print(classification_report(truth, preds_best, digits=4, zero_division=0, target_names=class_names))

    majority_class = int(torch.mode(labels[train_m]).values.item())
    baseline_preds = torch.full_like(truth, fill_value=majority_class)
    baseline_acc = accuracy_score(truth, baseline_preds) * 100
    baseline_f1_macro = f1_score(truth, baseline_preds, average="macro", zero_division=0) * 100

    recalls_per_class = recall_score(
        truth,
        preds_best,
        average=None,
        labels=list(range(n_classes)),
        zero_division=0,
    )

    model_acc = accuracy_score(truth, preds_best) * 100
    model_f1_macro = f1_score(truth, preds_best, average="macro", zero_division=0) * 100

    print(f"[BASELINE] Majority class={majority_class} ({class_names[majority_class] if majority_class < len(class_names) else majority_class})")
    print(f"[BASELINE] Acc={baseline_acc:.2f}% | Macro-F1={baseline_f1_macro:.2f}%")
    print(f"[MODEL]    Acc={model_acc:.2f}% | Macro-F1={model_f1_macro:.2f}%")

    print("Recall per class (test):")
    for i, r in enumerate(recalls_per_class):
        cname = class_names[i] if i < len(class_names) else f"Class {i}"
        print(f"  - {cname}: {r*100:.2f}%")

    epochs_done = range(1, len(hist["loss_tr"]) + 1)

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_done, hist["loss_tr"], label="Train Loss")
    plt.plot(epochs_done, hist["loss_val"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss - {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_done, hist["acc_tr"], label="Train Accuracy (%)")
    plt.plot(epochs_done, hist["acc_val"], label="Validation Accuracy (%)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    plt.title(f"Accuracy - {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs_done, hist["f1_val"], label="Validation Macro-F1 (%)", color="purple")
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1 (%)")
    plt.ylim(0, 100)
    plt.title(f"Validation Macro-F1 - {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


    # ── Overfitting check ─────────────────────────────────────────────
    print("\n[OVERFITTING CHECK]")

    # 1. Gap train vs val (al final del entrenamiento)
    final_acc_tr  = hist["acc_tr"][-1]
    final_acc_val = hist["acc_val"][-1]
    gap = final_acc_tr - final_acc_val
    print(f"  Train acc final : {final_acc_tr:.2f}%")
    print(f"  Val   acc final : {final_acc_val:.2f}%")
    print(f"  Gap             : {gap:.2f}%  {'⚠ posible overfitting' if gap > 10 else '✓ OK'}")

    # 2. Val F1 vs Test F1 (deben ser cercanos si el split es representativo)
    f1_gap = abs(best_macro_f1 * 100 - model_f1_macro)
    print(f"  Best val macro-F1 : {best_macro_f1*100:.2f}%")
    print(f"  Test macro-F1     : {model_f1_macro:.2f}%")
    print(f"  Diferencia        : {f1_gap:.2f}%  {'⚠ split sospechoso' if f1_gap > 5 else '✓ OK'}")

    # 3. Sanity check: predicción con etiquetas mezcladas en train
    # Si el modelo aprendió patrones reales, al mezclar las etiquetas del
    # train set y re-evaluar sobre esas mismas aristas, el score debe caer
    # cerca del azar (~33% para 3 clases).
    with torch.no_grad():
        logits_tr = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
        preds_tr_final = logits_tr[train_m].argmax(1).cpu()
        real_labels_tr = labels[train_m].cpu()
        shuffled_labels_tr = real_labels_tr[torch.randperm(len(real_labels_tr))]

    acc_real     = accuracy_score(real_labels_tr, preds_tr_final)
    acc_shuffled = accuracy_score(shuffled_labels_tr, preds_tr_final)
    expected_random = 1.0 / n_classes
    print(f"\n  Acc sobre train (etiquetas reales)   : {acc_real*100:.2f}%")
    print(f"  Acc sobre train (etiquetas mezcladas): {acc_shuffled*100:.2f}%")
    print(f"  Azar esperado                        : {expected_random*100:.1f}%")
    print(f"  {'✓ El modelo usa patrones reales' if acc_shuffled < expected_random + 0.05 else '⚠ Posible memorización'}")


## Caso 2: Enfoque end-to-end utilizando grafo con atributos de PeeringDB y muestreo aleatorio de vecinos (Random Neighbour Sampling)

In [ ]:

from modules.gnn_models import (
    GCN2L,
    GraphSAGE2L,
    GAT2L,
    GCN3L,
    GraphSAGE3L,
    GAT3L,
    GCNSampler2L,
    GraphSAGESample2L,
    GATSample2L
)

models = {
    # "GCN-Sampler":  GCNSampler2L,
    "SAGE-Sampler": GraphSAGESample2L,
    "GAT-Sampler":  GATSample2L
}


N_NEIGHBORS = 15

In [ ]:
# Cargar grafo con todos los atributos de nodos
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))

# Split estratificado sin fuga entre direcciones opuestas (store_eids=True por defecto)
gnn.split_edges_classification(
    train_size=0.8,
    val_size=0.15,
    seed=42,
    balance_mode="strict_equal",
)

g0 = gnn.graph
print("\n[Resumen del grafo]")
print(f"- Nodos:              {g0.num_nodes:,}")
print(f"- Aristas:            {g0.num_edges:,}")
print(f"- Dimension features: {g0.x.shape[1]}")
print(f"- Train edges:        {gnn.train_eids.shape[0]:,}")
print(f"- Val edges:          {gnn.val_eids.shape[0]:,}")
print(f"- Test edges:         {gnn.test_eids.shape[0]:,}")
labels = g0.edge_label
print(f"- Distribucion train: {Counter(labels[gnn.train_eids].tolist())}")
print(f"- Distribucion val:   {Counter(labels[gnn.val_eids].tolist())}")
print(f"- Distribucion test:  {Counter(labels[gnn.test_eids].tolist())}")


In [ ]:
device     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1024

# LinkNeighborLoader: 2 capas, N_NEIGHBORS vecinos por capa, muestreo a partir de aristas semilla
# (equivale a NeighborSampler + as_edge_prediction_sampler de DGL)
from torch_geometric.loader import LinkNeighborLoader

g = gnn.graph
edge_attr_all = g.edge_attr   # edge_feat de cada arista (se indexa por eid de las aristas semilla)

# DataLoaders sobre edge IDs (no node IDs)
train_loader = LinkNeighborLoader(
    g, num_neighbors=[N_NEIGHBORS, N_NEIGHBORS],
    edge_label_index=g.edge_index[:, gnn.train_eids], edge_label=g.edge_label[gnn.train_eids],
    batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)

val_loader = LinkNeighborLoader(
    g, num_neighbors=[N_NEIGHBORS, N_NEIGHBORS],
    edge_label_index=g.edge_index[:, gnn.val_eids], edge_label=g.edge_label[gnn.val_eids],
    batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

test_loader = LinkNeighborLoader(
    g, num_neighbors=[N_NEIGHBORS, N_NEIGHBORS],
    edge_label_index=g.edge_index[:, gnn.test_eids], edge_label=g.edge_label[gnn.test_eids],
    batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Device:        {device}")
print(f"Batch size:    {BATCH_SIZE}  |  Fan-out: {[N_NEIGHBORS, N_NEIGHBORS]}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")


In [ ]:
from tqdm.auto import tqdm
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

in_feats  = gnn.graph.x.shape[1]
n_classes = 3

# Degree features para forzar la asimetría C2P vs P2C en el decoder
g = gnn.graph
g.log_out_deg = torch.log1p(degree(g.edge_index[0], g.num_nodes).float())
g.log_in_deg  = torch.log1p(degree(g.edge_index[1], g.num_nodes).float())

for name, Model in models.items():
    print(f"\n===== Training {name} =====")

    model = Model(in_feats, 
                  GLOBAL_CONFIG['hidden_dim'], 
                  GLOBAL_CONFIG['out_dim'], out_feats_mlp=n_classes).to(device)
    optim     = torch.optim.Adam(model.parameters(), lr=GLOBAL_CONFIG['lr'], weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, 'min', patience=8, factor=0.5)

    hist = {"loss_tr": [], "acc_tr": [], "loss_val": [], "acc_val": []}
    best_loss, best_state, wait = 1e9, None, 0

    t0 = time.perf_counter()
    for epoch in range(1, 15 + 1):

        # ── TRAIN ──────────────────────────────────────────────────────
        model.train()
        loss_sum = corr = tot = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch:03d}", leave=False):
            batch  = batch.to(device)
            x      = batch.x                           # input node features
            labels = batch.edge_label.long()           # batch edge labels
            ea     = edge_attr_all[gnn.train_eids[batch.input_id.cpu()]].to(device)  # edge_feat aristas semilla

            h      = model.encode(x, batch.edge_index)                 # embeddings via neighbor sampling
            logits = model.decodeMLP(batch.edge_label_index, h, ea)    # edge logits (batch_size, 3)
            loss   = focal_loss(logits, labels, gamma=GLOBAL_CONFIG['focal_gamma'])

            optim.zero_grad()
            loss.backward()
            optim.step()

            loss_sum += loss.item() * len(labels)
            corr     += (logits.argmax(1) == labels).sum().item()
            tot      += len(labels)

        loss_tr = loss_sum / tot if tot > 0 else 0.0
        acc_tr  = 100 * corr / tot if tot > 0 else 0.0

        # ── VALIDATION ─────────────────────────────────────────────────
        model.eval()
        loss_sum = corr = tot = 0

        with torch.no_grad():
            for batch in val_loader:
                batch  = batch.to(device)
                x      = batch.x
                labels = batch.edge_label.long()
                ea     = edge_attr_all[gnn.val_eids[batch.input_id.cpu()]].to(device)

                h      = model.encode(x, batch.edge_index)
                logits = model.decodeMLP(batch.edge_label_index, h, ea)

                loss_sum += F.cross_entropy(logits, labels).item() * len(labels)
                corr     += (logits.argmax(1) == labels).sum().item()
                tot      += len(labels)

        loss_val = loss_sum / tot if tot > 0 else 0.0
        acc_val  = 100 * corr / tot if tot > 0 else 0.0

        scheduler.step(loss_val)

        hist['loss_tr'].append(loss_tr);   hist['acc_tr'].append(acc_tr)
        hist['loss_val'].append(loss_val); hist['acc_val'].append(acc_val)

        # Early stopping based on validation loss
        if loss_val < best_loss - 1e-4:
            best_loss, wait = loss_val, 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= GLOBAL_CONFIG['patience']:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 5 == 0 or epoch == 1:
            print(f"ep{epoch:03d} | loss_tr {loss_tr:.4f}  acc_tr {acc_tr:.2f}% | "
                  f"loss_val {loss_val:.4f}  acc_val {acc_val:.2f}%")

    # ── FINAL EVALUATION (test) ────────────────────────────────────────
    tiempo_total = time.perf_counter() - t0
    mins, secs = divmod(tiempo_total, 60)
    print(f"\n{'='*50}")
    print(f"[TIEMPO] {name} (NeighborSampling): {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
    print(f"{'='*50}")

    model.load_state_dict(best_state)
    model.eval()
    preds_all, truth_all = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch  = batch.to(device)
            x      = batch.x
            labels = batch.edge_label.long()
            ea     = edge_attr_all[gnn.test_eids[batch.input_id.cpu()]].to(device)

            h      = model.encode(x, batch.edge_index)
            logits = model.decodeMLP(batch.edge_label_index, h, ea)

            preds_all.append(logits.argmax(1).cpu())
            truth_all.append(labels.cpu())

    preds = torch.cat(preds_all)
    truth = torch.cat(truth_all)

    cm = confusion_matrix(truth, preds)
    plot_confusion_matrix_2(cm, ["P2P","C2P","P2C"], normalize=False, title=f"Confusion - {name}")
    plot_confusion_matrix_2(cm, ["P2P","C2P","P2C"], normalize=True,  title=f"Confusion (%) - {name}")
    print(classification_report(truth, preds, digits=4, target_names=["P2P","C2P","P2C"]))

    # ── Training curves ────────────────────────────────────────────────
    ep_axis = range(1, len(hist['loss_tr']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(ep_axis, hist['loss_tr'],  label='Train Loss')
    axes[0].plot(ep_axis, hist['loss_val'], label='Validation Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Loss - {name}'); axes[0].legend()

    axes[1].plot(ep_axis, hist['acc_tr'],  label='Train Accuracy (%)')
    axes[1].plot(ep_axis, hist['acc_val'], label='Validation Accuracy (%)')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0, 100)
    axes[1].set_title(f'Accuracy - {name}'); axes[1].legend()

    plt.tight_layout(); plt.show()


## Caso 3: Se utiliza grafo con atributos de PeeringDB y muestreo basado en comunidades (ClusterGCN)

In [ ]:

from modules.gnn_models import (
    GCN2L,
    GraphSAGE2L,
    GAT2L,
    GCN3L,
    GraphSAGE3L,
    GAT3L,
    GCNSampler2L,
    GraphSAGESample2L,
    GATSample2L
)

models = {
    "GCN-Sampler":  GCNSampler2L,
    "SAGE-Sampler": GraphSAGESample2L,
    "GAT-Sampler":  GATSample2L
}


In [ ]:
# 1. Load data with the new format (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))

gnn.split_edges_classification(
    train_size=0.8,
    val_size=0.15,
    seed=42,
    balance_mode="strict_equal",   # use "proportional" to keep natural class distribution
)

g0 = gnn.graph
print("\n[Loaded graph summary]")
print(f"- Nodes: {g0.num_nodes:,}")
print(f"- Edges: {g0.num_edges:,}")
print(f"- Feature dimension: {g0.x.shape[1]}")
print(f"- Edge data keys: {[k for k in g0.keys() if g0.is_edge_attr(k)]}")
print(f"- Train edges: {g0.train_mask.sum().item():,}")
print(f"- Validation edges: {g0.val_mask.sum().item():,}")
print(f"- Test edges: {g0.test_mask.sum().item():,}")
if getattr(g0, "edge_label", None) is not None:
    print(f"- Global label distribution: {Counter(g0.edge_label.tolist())}")
    tr = g0.edge_label[g0.train_mask]
    va = g0.edge_label[g0.val_mask]
    te = g0.edge_label[g0.test_mask]
    print(f"- Train distribution: {Counter(tr.tolist())}")
    print(f"- Validation distribution: {Counter(va.tolist())}")
    print(f"- Test distribution: {Counter(te.tolist())}")


In [ ]:
num_parts = 1000
device    = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1024

# ClusterData + ClusterLoader (equivale a ClusterGCNSampler de DGL).
# Los atributos de arista (label, máscaras, edge_attr) se reordenan junto con el subgrafo.
from torch_geometric.loader import ClusterData, ClusterLoader

cluster_data = ClusterData(gnn.graph, num_parts=num_parts)

train_loader = ClusterLoader(
    cluster_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False)

val_loader = ClusterLoader(
    cluster_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False)

test_loader = ClusterLoader(
    cluster_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False)

print(f"Device:      {device}")
print(f"Clusters:    {num_parts}  |  Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}")


In [ ]:
from tqdm.auto import tqdm
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

in_feats  = gnn.graph.x.shape[1]
n_classes = 3

# Degree features para forzar la asimetría C2P vs P2C en el decoder
g = gnn.graph
g.log_out_deg = torch.log1p(degree(g.edge_index[0], g.num_nodes).float())
g.log_in_deg  = torch.log1p(degree(g.edge_index[1], g.num_nodes).float())

for name, Model in models.items():
    print(f"\n===== Training {name} =====")

    model = Model(in_feats, 
                  GLOBAL_CONFIG['hidden_dim'], 
                  GLOBAL_CONFIG['out_dim'], 
                  out_feats_mlp=n_classes).to(device)
    
    optim    = torch.optim.Adam(model.parameters(), lr=GLOBAL_CONFIG['lr'], weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, 'min', patience=8, factor=0.5)

    hist = {"loss_tr": [], "acc_tr": [], "loss_val": [], "acc_val": []}
    best_loss, best_state, wait = 1e9, None, 0

    t0 = time.perf_counter()
    for epoch in range(1, 15 + 1):

        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()
        loss_sum = corr = tot = 0

        for subg in tqdm(train_loader, desc=f"Epoch {epoch:03d}", leave=False):
            if subg.num_edges == 0:
                continue
            subg = subg.to(device)
            x = subg.x.float()
            y = subg.edge_label.long()
            m = subg.train_mask.bool()
            if m.sum() == 0:
                continue

            h      = model.encode(x, subg.edge_index)
            logits = model.decodeMLP(subg.edge_index, h, subg.edge_attr)
            loss   = focal_loss(logits[m], y[m], gamma=GLOBAL_CONFIG['focal_gamma'])

            optim.zero_grad()
            loss.backward()
            optim.step()

            loss_sum += loss.item() * m.sum().item()
            corr     += (logits[m].argmax(1) == y[m]).sum().item()
            tot      += m.sum().item()

        loss_tr = loss_sum / tot if tot > 0 else 0.0
        acc_tr  = 100 * corr / tot if tot > 0 else 0.0

        # ── VALIDATION ─────────────────────────────────────────────────────
        model.eval()
        loss_sum = corr = tot = 0

        with torch.no_grad():
            for subg in val_loader:
                if subg.num_edges == 0:
                    continue
                subg = subg.to(device)
                x = subg.x.float()
                y = subg.edge_label.long()
                m = subg.val_mask.bool()
                if m.sum() == 0:
                    continue

                h      = model.encode(x, subg.edge_index)
                logits = model.decodeMLP(subg.edge_index, h, subg.edge_attr)

                loss_sum += F.cross_entropy(logits[m], y[m]).item() * m.sum().item()
                corr     += (logits[m].argmax(1) == y[m]).sum().item()
                tot      += m.sum().item()

        loss_val = loss_sum / tot if tot > 0 else 0.0
        acc_val  = 100 * corr / tot if tot > 0 else 0.0

        scheduler.step(loss_val)

        hist['loss_tr'].append(loss_tr);   hist['acc_tr'].append(acc_tr)
        hist['loss_val'].append(loss_val); hist['acc_val'].append(acc_val)

        # Early stopping based on validation loss
        if loss_val < best_loss - 1e-4:
            best_loss, wait = loss_val, 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= GLOBAL_CONFIG['patience']:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 5 == 0 or epoch == 1:
            print(f"ep{epoch:03d} | loss_tr {loss_tr:.4f}  acc_tr {acc_tr:.2f}% | "
                  f"loss_val {loss_val:.4f}  acc_val {acc_val:.2f}%")

    # ── FINAL EVALUATION (test) ────────────────────────────────────────────
    tiempo_total = time.perf_counter() - t0
    mins, secs = divmod(tiempo_total, 60)
    print(f"\n{'='*50}")
    print(f"[TIEMPO] {name} (ClusterGCN): {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
    print(f"{'='*50}")

    model.load_state_dict(best_state)
    model.eval()
    preds_all, truth_all = [], []

    with torch.no_grad():
        for subg in test_loader:
            if subg.num_edges == 0:
                continue
            subg = subg.to(device)
            x = subg.x.float()
            y = subg.edge_label.long()
            m = subg.test_mask.bool()
            if m.sum() == 0:
                continue

            h      = model.encode(x, subg.edge_index)
            logits = model.decodeMLP(subg.edge_index, h, subg.edge_attr)

            preds_all.append(logits[m].argmax(1).cpu())
            truth_all.append(y[m].cpu())

    preds = torch.cat(preds_all)
    truth = torch.cat(truth_all)

    cm = confusion_matrix(truth, preds)
    plot_confusion_matrix_2(cm, ["P2P","C2P","P2C"], normalize=False, title=f"Confusion - {name}")
    plot_confusion_matrix_2(cm, ["P2P","C2P","P2C"], normalize=True,  title=f"Confusion (%) - {name}")
    print(classification_report(truth, preds, digits=4, target_names=["P2P","C2P","P2C"]))

    # ── Training curves ────────────────────────────────────────────────────
    ep_axis = range(1, len(hist['loss_tr']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(ep_axis, hist['loss_tr'],  label='Train Loss')
    axes[0].plot(ep_axis, hist['loss_val'], label='Validation Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Loss - {name}'); axes[0].legend()

    axes[1].plot(ep_axis, hist['acc_tr'],  label='Train Accuracy (%)')
    axes[1].plot(ep_axis, hist['acc_val'], label='Validation Accuracy (%)')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0, 100)
    axes[1].set_title(f'Accuracy - {name}'); axes[1].legend()

    plt.tight_layout(); plt.show()


## Caso 4: Se utiliza grafo con atributos de nodos generados aleatoriamente

In [ ]:
models = {
    'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}



# 1. Load data with the new format (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))


gnn.add_random_features(dim=64, std=0.05, seed=42, mode="uniform")


gnn.split_edges_classification(
    train_size=0.8,
    val_size=0.15,
    seed=42,
    balance_mode="strict_equal",   # use "proportional" to keep natural class distribution
)

g0 = gnn.graph

In [ ]:
import torch, torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Parámetros 
# ------------------------------------------
in_feats  = gnn.graph.x.shape[1]
device    = "cuda" if torch.cuda.is_available() else "cpu"
lr        = GLOBAL_CONFIG['lr']
patience  = GLOBAL_CONFIG['patience']
n_classes = 3

# Datos y máscaras 
# ------------------------------------------
g       = gnn.graph.to(device)
feat    = g.x.to(device)
edge_index = g.edge_index.to(device)
edge_attr  = g.edge_attr.to(device) if getattr(g, "edge_attr", None) is not None else None
label_key = next((k for k in ["edge_label", "label", "relationship", "Relationship"] if getattr(g, k, None) is not None), None)
labels  = getattr(g, label_key).long().to(device)
train_m = g.train_mask.to(device)
test_m  = g.test_mask.to(device)

for name, Model in models.items():
    print(f"\n Training {name} \n ---------------")

    model = Model(in_feats, GLOBAL_CONFIG['hidden_dim'], GLOBAL_CONFIG['out_dim'], out_feats_mlp=n_classes).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=GLOBAL_CONFIG['weight_decay'])

    # histórico por modelo
    hist = {"loss_tr": [], "acc_tr": [], "loss_te": [], "acc_te": []}
    best_loss, wait = 1e9, 0

    t0 = time.perf_counter()
    for ep in range(1, GLOBAL_CONFIG['epochs'] + 1):
        # Entrenamiento 
        # ----------
        model.train()
        h      = model.encode(feat, edge_index)
        logits = model.decodeMLP(edge_index, h, edge_attr)                  # (E, 3)

        loss   = F.cross_entropy(logits[train_m], labels[train_m])
        # loss   = F.cross_entropy(logits[train_m], labels[train_m], weight=class_weights)

        
        opt.zero_grad() 
        loss.backward()
        opt.step()

        with torch.no_grad():
            preds_tr = logits[train_m].argmax(1)
            acc_tr   = (preds_tr == labels[train_m]).float().mean().item()

        # ---------- test ----------
        model.eval()
        with torch.no_grad():
            logits_t = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
            loss_te  = F.cross_entropy(logits_t[test_m], labels[test_m])
            # loss_te  = F.cross_entropy(logits_t[test_m], labels[test_m], weight=class_weights)
            preds_t  = logits_t[test_m].argmax(1)
            acc_t    = (preds_t == labels[test_m]).float().mean().item()

        # guarda histórico
        hist["loss_tr"].append(loss.item())
        hist["acc_tr"].append(acc_tr * 100)
        hist["loss_te"].append(loss_te.item())
        hist["acc_te"].append(acc_t * 100)

        # early-stopping
        if loss_te < best_loss - 1e-4:
            best_loss, wait = loss_te, 0
            best_w = model.state_dict()
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stop en época {ep}")
                break

        if ep % 10 == 0 or ep == 1:
            print(f"ep{ep:02d} | loss_tr {loss:.4f} acc_tr {acc_tr:.2%} | "
                  f"loss_te {loss_te:.4f} acc_te {acc_t:.2%}")

    tiempo_total = time.perf_counter() - t0
    mins, secs = divmod(tiempo_total, 60)
    print(f"\n{'='*50}")
    print(f"[TIEMPO] {name}: {int(mins)}m {secs:.1f}s  ({tiempo_total:.1f}s total)")
    print(f"{'='*50}")

    # ---------- evaluación final ----------
    model.load_state_dict(best_w)
    with torch.no_grad():
        logits_best = model.decodeMLP(edge_index, model.encode(feat, edge_index), edge_attr)
        preds_best  = logits_best[test_m].argmax(1).cpu()
        truth       = labels[test_m].cpu()

    cm = confusion_matrix(truth, preds_best)
    plot_confusion_matrix_2(cm, ["P2P","C2P","P2C"], False,
                            title=f"Confusión – {name}")
    plot_confusion_matrix_2(cm, ["P2P","C2P","P2C"], True,
                            title=f"Confusión (%) – {name}")

    print(classification_report(truth, preds_best, digits=4))

    # ---------- curvas de entrenamiento ----------
    epochs_done = range(1, len(hist["loss_tr"]) + 1)

    plt.figure(figsize=(7,4))
    plt.plot(epochs_done, hist["loss_tr"], label="Loss train")
    plt.plot(epochs_done, hist["loss_te"], label="Loss test")
    plt.xlabel("Época"); plt.ylabel("Loss"); plt.title(f"Loss – {name}")
    plt.legend(); plt.tight_layout(); plt.show()

    plt.figure(figsize=(7,4))
    plt.plot(epochs_done, hist["acc_tr"], label="Acc train (%)")
    plt.plot(epochs_done, hist["acc_te"], label="Acc test (%)")
    plt.xlabel("Época"); plt.ylabel("Accuracy (%)"); plt.ylim(0,100)
    plt.title(f"Accuracy – {name}")
    plt.legend(); plt.tight_layout(); plt.show()


## PREGUNTAS:
* ¿Por que incluso con atributos aleatorios se obtiene una buena accuracy con la GCN?
* ¿Por qué usando random Neighbputhood se obtienen la mejor accuracy?¿Por qué ocurre eso?¿Qué nos dice?
* Al ocupar los attr creados aleatoriamente , opdemos ver q se van aprendindo de memoria y por lo msimo la acurracy ntre mas epochs se emṕieza a separara
* ¿Que informacion del grafo y las GNNS puedo sacar con estos rsulatdos?
* construir hatmap d correlacion entre atributos de perngdb y uno tmb para las centralidades

## Comparación: Todos los modelos
**2L vs 3L × GCN / GraphSAGE / GAT × Decoder MLP / Bilinear**

Entrena las 12 combinaciones con la misma configuración y genera una tabla comparativa.

In [ ]:
import torch, torch.nn.functional as F, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score

ALL_MODELS = {
    'GCN-2L':    GCN2L,
    'GCN-3L':    GCN3L,
    'SAGE-2L':   GraphSAGE2L,
    'SAGE-3L':   GraphSAGE3L,
    'GAT-2L':    GAT2L,
    'GAT-3L':    GAT3L,
}
ALL_DECODERS = ['MLP', 'Bilinear']

COMP_HIDDEN   = GLOBAL_CONFIG['hidden_dim']
COMP_OUT      = GLOBAL_CONFIG['out_dim']
COMP_LR       = GLOBAL_CONFIG['lr']
COMP_EPOCHS   = GLOBAL_CONFIG['epochs']
COMP_PATIENCE = GLOBAL_CONFIG['patience']


def _train_one(ModelClass, decoder_name, in_feats, n_cls, cw,
               g, feat, labels, train_m, val_m, test_m, device, extra_kw=None):
    if extra_kw is None:
        extra_kw = {}
    model  = ModelClass(in_feats, COMP_HIDDEN, COMP_OUT,
                        out_feats_mlp=n_cls, edge_dim=1,
                        drop=GLOBAL_CONFIG['dropout'], **extra_kw).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=COMP_LR,
                              weight_decay=GLOBAL_CONFIG['weight_decay'])
    sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)
    edge_index = g.edge_index
    edge_attr  = getattr(g, 'edge_attr', None)
    if decoder_name == 'Bilinear':
        decode = lambda h: model.decodeBilinear(edge_index, h)
    else:
        decode = lambda h: model.decodeMLP(edge_index, h, edge_attr)

    best_f1, best_loss, wait, best_w = -1.0, float("inf"), 0, None

    t0 = time.perf_counter()
    for ep in range(1, COMP_EPOCHS + 1):
        model.train()
        logits = decode(model.encode(feat, edge_index))
        loss = F.cross_entropy(logits[train_m], labels[train_m],
                               weight=cw, label_smoothing=GLOBAL_CONFIG['label_smooth'])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CONFIG['grad_clip'])
        opt.step(); sched.step(ep - 1 + 1 / COMP_EPOCHS)

        model.eval()
        with torch.no_grad():
            lv       = decode(model.encode(feat, edge_index))
            loss_val = F.cross_entropy(lv[val_m], labels[val_m])
            f1v      = f1_score(labels[val_m].cpu(), lv[val_m].argmax(1).cpu(),
                                average="macro", zero_division=0)

        if f1v > best_f1 + 1e-4 or (abs(f1v - best_f1) <= 1e-4 and loss_val < best_loss - 1e-4):
            best_f1, best_loss, wait, best_w = f1v, loss_val.item(), 0, model.state_dict()
        else:
            wait += 1
            if wait >= COMP_PATIENCE:
                break
    tiempo_total = time.perf_counter() - t0

    model.load_state_dict(best_w or model.state_dict())
    model.eval()
    with torch.no_grad():
        preds = decode(model.encode(feat, edge_index))[test_m].argmax(1).cpu()
    truth = labels[test_m].cpu()

    recs = recall_score(truth, preds, labels=[0, 1, 2], average=None, zero_division=0) * 100
    return {
        "Acc%":     round(accuracy_score(truth, preds) * 100, 2),
        "F1-macro": round(f1_score(truth, preds, average="macro",    zero_division=0) * 100, 2),
        "F1-w":     round(f1_score(truth, preds, average="weighted", zero_division=0) * 100, 2),
        "P2P-rec":  round(recs[0], 2),
        "C2P-rec":  round(recs[1], 2),
        "P2C-rec":  round(recs[2], 2),
        "Time (s)": round(tiempo_total, 2),
    }


# ── Setup ────────────────────────────────────────────────────────────────────────
device     = "cuda" if torch.cuda.is_available() else "cpu"
in_feats_c = gnn.graph.x.shape[1]
g_c        = gnn.graph.to(device)
feat_c     = g_c.x.to(device)

lk = next((k for k in ["edge_label", "label", "relationship", "Relationship"] if getattr(g_c, k, None) is not None), None)
labels_c = getattr(g_c, lk).long().to(device)
train_mc = g_c.train_mask.bool().to(device)
val_mc   = g_c.val_mask.bool().to(device)
test_mc  = g_c.test_mask.bool().to(device)

n_cls_c = int(labels_c[train_mc | val_mc | test_mc].max().item()) + 1
cc      = torch.bincount(labels_c[train_mc], minlength=n_cls_c).float()
cw_c    = (cc.sum() / (n_cls_c * cc.clamp(min=1.0))).to(device)

# ── Loop ─────────────────────────────────────────────────────────────────────────
rows = []
for mname, MCls in ALL_MODELS.items():
    for dec in ALL_DECODERS:
        print(f"▶ {mname:10s} + {dec:8s} ...", end=" ", flush=True)
        extra_kw = {'num_heads': GLOBAL_CONFIG['num_heads']} if 'GAT' in mname else {}
        try:
            m = _train_one(MCls, dec, in_feats_c, n_cls_c, cw_c,
                           g_c, feat_c, labels_c, train_mc, val_mc, test_mc, device, extra_kw=extra_kw)
            rows.append({"Modelo": mname, "Decoder": dec, **m})
            print(f"Acc={m['Acc%']:.1f}%  F1={m['F1-macro']:.1f}%  Time={m['Time (s)']:.1f}s")
            print(f"Modelo {mname} finalizado en {m['Time (s)']:.2f} segundos")
        except Exception as e:
            rows.append({"Modelo": mname, "Decoder": dec,
                         "Acc%": None, "F1-macro": None, "F1-w": None,
                         "P2P-rec": None, "C2P-rec": None, "P2C-rec": None,
                         "Time (s)": None})
            print(f"ERROR: {e}")

# ── Tabla ─────────────────────────────────────────────────────────────────────────
df_cmp = pd.DataFrame(rows).set_index(["Modelo", "Decoder"]).sort_values("F1-macro", ascending=False)
print()
print("══ Comparación completa ═══════════════════════════════════════════════════════════")
print(df_cmp.to_string())